# Patrón Estructural: Composite

## Introducción
El patrón Composite permite componer objetos en estructuras de árbol y trabajar con esas estructuras como si fueran objetos individuales.

## Objetivos
- Comprender cómo agrupar objetos en jerarquías.
- Identificar cuándo es útil el patrón Composite.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: App de Gestión de Archivos**
Un sistema de archivos puede contener carpetas y archivos. El patrón Composite permite tratar carpetas y archivos de la misma manera, facilitando operaciones recursivas como calcular el tamaño total.

**¿Dónde se usa en proyectos reales?**
En sistemas de archivos, menús de aplicaciones, estructuras organizacionales, etc.

## Sin patrón Composite (forma errónea)
El código cliente debe distinguir entre archivos y carpetas, lo que complica la lógica.

In [1]:
class Archivo:
    def __init__(self, nombre: str, tamano: int) -> None:
        self.nombre = nombre
        self.tamano = tamano

class Carpeta:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.elementos: list = []
    def agregar(self, elemento: object) -> None:
        self.elementos.append(elemento)

# El cliente debe recorrer y distinguir manualmente

## Con patrón Composite (forma correcta)
Ambos, archivos y carpetas, implementan la misma interfaz, simplificando la lógica del cliente.

In [2]:
import abc

class Elemento(abc.ABC):
    @abc.abstractmethod
    def mostrar(self) -> None:
        ...

class Archivo(Elemento):
    def __init__(self, nombre: str, tamano: int) -> None:
        self.nombre = nombre
        self.tamano = tamano
    def mostrar(self) -> None:
        print(f'Archivo: {self.nombre} ({self.tamano} KB)')

class Carpeta(Elemento):
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.elementos: list[Elemento] = []
    def agregar(self, elemento: Elemento) -> None:
        self.elementos.append(elemento)
    def mostrar(self) -> None:
        print(f'Carpeta: {self.nombre}')
        for e in self.elementos:
            e.mostrar()

raiz = Carpeta('raiz')
raiz.agregar(Archivo('doc.txt', 10))
raiz.mostrar()

Carpeta: raiz
Archivo: doc.txt (10 KB)


## UML del patrón Composite
```plantuml
@startuml
abstract class Elemento {
    + mostrar()
}
class Archivo {
    + mostrar()
}
class Carpeta {
    + agregar(elemento)
    + mostrar()
}
Elemento <|-- Archivo
Elemento <|-- Carpeta
Carpeta *-- Elemento
@enduml
```

## Otro ejemplo de la vida real: Carrito de compras con combos anidados
**Contexto:** en una tienda online, un carrito puede tener productos individuales y **combos** (bundles) que agrupan varios productos con descuento — y un combo puede a su vez contener otro combo (ej. un "Combo Mega" que incluye el "Combo Verano" más una toalla). Calcular el precio total del carrito debe funcionar sin importar cuántos niveles de anidamiento tenga.

### Sin patrón (forma errónea)
El código que calcula el total debe distinguir con `isinstance` entre producto y combo, y solo contempla un nivel: si un combo contiene otro combo, el cálculo queda incompleto.

In [3]:
class ProductoSimple:
    def __init__(self, nombre: str, precio: float) -> None:
        self.nombre = nombre
        self.precio = precio

class Combo:
    def __init__(self, nombre: str, productos: list) -> None:
        self.nombre = nombre
        self.productos = productos  # se asume que solo contiene ProductoSimple

def calcular_total(items: list) -> float:
    total = 0.0
    for item in items:
        if isinstance(item, ProductoSimple):
            total += item.precio
        elif isinstance(item, Combo):
            for p in item.productos:
                total += p.precio  # falla si p es a su vez un Combo
    return total

combo_verano = Combo('Combo Verano', [ProductoSimple('Gorra', 15000), ProductoSimple('Gafas', 20000)])
combo_mega = Combo('Combo Mega', [combo_verano, ProductoSimple('Toalla', 25000)])  # combo dentro de combo

carrito = [ProductoSimple('Camiseta', 30000), combo_mega]
# calcular_total(carrito)  # AttributeError: 'Combo' object has no attribute 'precio'

### Con patrón (forma correcta)
`Producto` y `Combo` implementan la misma interfaz `ItemCarrito.calcular_precio()`. Un `Combo` simplemente suma el precio de sus items, sin importar si cada item es un producto simple u otro combo — la recursión funciona sola.

In [4]:
import abc

class ItemCarrito(abc.ABC):
    @abc.abstractmethod
    def calcular_precio(self) -> float:
        ...

class Producto(ItemCarrito):
    def __init__(self, nombre: str, precio: float) -> None:
        self.nombre = nombre
        self.precio = precio
    def calcular_precio(self) -> float:
        return self.precio

class Combo(ItemCarrito):
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.items: list[ItemCarrito] = []
    def agregar(self, item: ItemCarrito) -> None:
        self.items.append(item)
    def calcular_precio(self) -> float:
        return sum(item.calcular_precio() for item in self.items)


carrito = [Producto('Camiseta', 30000)]

combo_verano = Combo('Combo Verano')
combo_verano.agregar(Producto('Gorra', 15000))
combo_verano.agregar(Producto('Gafas', 20000))

combo_mega = Combo('Combo Mega')
combo_mega.agregar(combo_verano)  # combo dentro de combo, sin problema
combo_mega.agregar(Producto('Toalla', 25000))

carrito.append(combo_mega)

total = sum(item.calcular_precio() for item in carrito)
print(f'Total del carrito: ${total}')

Total del carrito: $90000


### UML del ejemplo de carrito con combos
```plantuml
@startuml
abstract class ItemCarrito {
    + calcular_precio()
}
class Producto {
    + calcular_precio()
}
class Combo {
    + agregar(item)
    + calcular_precio()
}
ItemCarrito <|-- Producto
ItemCarrito <|-- Combo
Combo o-- ItemCarrito
@enduml
```

### ¿Dónde más se usa Composite?
- **Catálogos con bundles anidados:** exactamente este ejemplo — combos que contienen productos u otros combos (Amazon, Mercado Libre).
- **Menús de aplicaciones y de restaurantes:** una opción de menú simple o un submenú que a su vez contiene más opciones, renderizados con la misma lógica recursiva.
- **Estructuras organizacionales:** un empleado individual o un departamento (que contiene empleados y otros departamentos) para calcular nómina total o número de reportes directos.
- **Árboles de UI (DOM/widgets):** un componente visual simple o un contenedor que agrupa otros componentes, todos con la misma interfaz `renderizar()`.
- **Sistemas de archivos:** carpetas que contienen archivos u otras carpetas (el ejemplo con el que abre este notebook).

**Ejercicio de reflexión:** en la versión "con patrón", ¿qué pasaría si `Combo` necesitara aplicar un 10% de descuento sobre el total de sus items? ¿Dónde agregarías esa lógica sin tener que tocar `Producto`?

## Actividad
Crea tu propio Composite para modelar una estructura organizacional (departamentos y empleados).

---
## Explicación de conceptos clave
- **Jerarquía recursiva:** Permite tratar objetos individuales y compuestos de la misma manera.
- **Simplicidad para el cliente:** El cliente no necesita distinguir entre tipos de elementos.
- **Aplicación en la vida real:** Útil en sistemas de archivos, menús y estructuras organizacionales.

## Conclusión
El patrón Composite es ideal para trabajar con estructuras jerárquicas y recursivas. Facilita la extensión y el mantenimiento en sistemas complejos.